## Домашнее задание 7

### Задача 1

Выяснилось, что задача с придумыванием примера системы, не обладающей глобальной асимптотической устойчивостью и (не) функции Ляпунова без радиальной неограниченности, сложна.

Поэтому теперь задача состоит в том, чтобы осознать пример из книги Халила. Критерий "осознавания" - что вы прям поняли, в чем суть. До достижения этого момента надо крутить пример так и сяк, советоваться с одногруппниками, думать, задавать себе вопросы, которые лежат рядом с этой темой. Например:
- для чего в сущности нужна радиальная неограниченность? Как она участвует в доказательстве?
- система в примере Халила не обладает глобальной устойчивостью, но обладает ли она локальной?
- асимптотически ли она устойчива?

Ну и так далее. Надо разобраться

### Задача 2

Будем работать с оценкой параметров линейной системы. На паре мы вывели оценку для траектории, которая доступна целиком, а теперь сделаем то же самое, только будем добавлять к текущей оценке поправку, умноженную на небольшое число (так называемый learning rate, скорость обучения).

Изучите код в следующей клетке. Осознайте, что и почему делается, при необходимости повторите вывод градиентного шага с помощью https://en.wikipedia.org/wiki/Matrix_calculus.

Матрица оценки изменяется на каждом шаге в направлении, противоположном градиенту функции ошибки предсказания поведения системы.

Попробуйте поменять скорость обучения, начальное приближение, саму матрицу, число итераций алгоритма.

Обратите внимание, что состояние системы генерируется случайным образом на каждом шаге. Здесь мы имеем дело не с траекторией, а с набором состояний и предсказанных по модели для них следующих.

In [ ]:
import numpy as np

A = np.array([[1.0, 2],
              [3, 4]])

A_hat = np.array([[0.0, 0],
               [0, 0]])

for i in range(300):
    x = np.array([[np.random.randn()],
                  [np.random.randn()]])
    
    xp = A @ x
    
    A_hat -= 0.01 * (-2 * xp @ x.T + 2 * A_hat @ x @ x.T)

print(A_hat)

### Задача 3

Примените этот метод оценки к матрице осциллятора без управления.

Есть место, на котором можно подскользнуться. Функция `oscillator.get_state()` возвращает состояние, а не его копию, так что если состояние изменится, то и переменная, в которую оно сохранено, тоже. Это можно обойти с помощью `copy.deepcopy()`.

In [ ]:
import numpy as np
import math
import cv2
import matplotlib.pyplot as plt
from ipywidgets import interact
from IPython.display import display, clear_output
import copy

def plot_1d(data, xlabel, ylabel, title):
    plt.figure(figsize=(15, 4))
    
    for d in data:
        plt.plot(d)
    
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)

    plt.show()

class Oscillator:
    def __init__(self, A, m = 1.0, x0 = 3, v0 = 0.01,
                 dt = 0.05, real_stiffness = 0.7,
                 stiffness_initial_guess = 0.37):
        self.m = m
        
        self.x = np.array([[x0], [v0]])
        self.dt = dt
        
        self.real_stiffness = real_stiffness
        self.stiffness_estimate = stiffness_initial_guess
        
        self.A = A
        
        self.B = np.array([[0],
                           [1 / self.m]])
        
    def get_state(self):
        return self.x
    
    def sys_dyn(self, u_scalar):
        x_dot = np.zeros_like(self.x)
        
        u = np.array([[u_scalar]])
        
        x_dot = self.A @ self.x + self.B @ u
        
        return x_dot
    
    def integrate_sys_dyn(self, sys_dyn):        
        self.x[1] += sys_dyn[1] * self.dt
        self.x[0] += self.x[1] * self.dt

    def draw(self, canvas, WIND_X, WIND_Y, scale, color = (123, 234, 234)):
        cv2.line(canvas, (0, WIND_Y // 2), (WIND_X, WIND_Y // 2), (123, 234, 234), 5)
        
        cv2.rectangle((WIND_X // 2 + self.x[0, 0] * scale - 25, WIND_Y // 2 - 20),
                      (WIND_X // 2 + self.x[0, 0] * scale + 25, WIND_Y // 2),
                      color, 7)

def run_oscillator(x0, iter_num, visualize, m, real_stiffness, dt, compensation = False,
                 update = False):
    if (visualize == True):
        WIND_X = 700
        WIND_Y = 700
        canvas = np.ones((700, 700, 3), np.uint8) * 70
    
    #real A matrix
    A = np.array([[0, 1],
                  [- real_stiffness / m, 0]])
    
    #initial estimate
    A_hat = np.array([[0.0, 0],
                      [0, 0]])

    i = 0

    x_traj = []
    v_traj = []
    u_traj = []
    
    oscillator = Oscillator(A, x0 = x0, dt = dt, real_stiffness = real_stiffness)
    
    desired_x = 0
            
    while(True):
        state = copy.deepcopy(oscillator.get_state())
        
        control = 0
        
        x_dot = oscillator.sys_dyn(control)
                
        oscillator.integrate_sys_dyn(x_dot)
        
        # YOUR CODE BELOW
        
        

        # YOUR CODE ABOVE
        
        x_traj.append(state[0, 0])
        v_traj.append(state[1, 0])
        u_traj.append(control)
        
        i += 1

        if (i > iter_num):
            break

        if (visualize == True):
            canvas = cv2.addWeighted(canvas, 0.93, canvas, 0, 0)

            oscillator.draw(canvas, WIND_X // 2, WIND_Y // 2, 100,
                          state[0, 0], (123, 234, 45), inverted = False)

            cv2.imshow("oscillator", canvas)

            key = cv2.waitKey(10) & 0xFF

            if (key == ord('q')):
                break
        
    if (visualize == True):
        cv2.destroyAllWindows()
        cv2.waitKey(10)
    
    plot_1d([x_traj], "time", "x", "coordinate by time")
    plot_1d([v_traj], "time", "x_dot", "velocity by time")
    plot_1d([u_traj], "time", "F", "force by time")
    
    plt.figure(figsize=(10, 10))
    plt.plot(x_traj, v_traj)
    plt.show()
    clear_output(wait = True)
    
    return x_traj, v_traj, u_traj, A_hat, A

dt = 0.05

x_hist, v_hist, u_hist, A_hat, A = run_oscillator(x0 = -2, iter_num = 1200, visualize = False,
                        m = 1, real_stiffness = 0.9, dt = dt, compensation = True, update = True)

In [ ]:
print(A_hat)

Похоже ли это на матрицу, которая используется при нахождении вектора x_dot в интегрировании системы?

Как будто бы это не она. Матрица $A$ сама по себе выглядит так:

In [ ]:
print(A)

Чем объясняется это различие?

Давайте вспомним, как связаны матрицы непрерывной и дискретной систем. Именно ответ на этот вопрос позволит получить из A_hat в коде оценку на матрицу непрерывной системы. Сделайте это в следующей клетке.

In [ ]:
# YOUR CODE BELOW



# YOUR CODE ABOVE

print(A_estimate_continious)

In [ ]:
print(np.allclose(A, A_estimate_continious, atol=0.1))

Если здесь True, то ура, задача решена корректно.

### Задача 4

Давайте обобщим эту технику на системы с управлением. Вместо системы $\dot{x} = A x + B u$ рассмотрим систему $\dot{x} = C y$, где $y$ - дополненный вектор состояния, а $C$ - дополненная матрица системы:

$C = (A|B)$

$y = 
\begin{pmatrix}
x \\
u
\end{pmatrix}
$

Будем оценивать матрицу $C$ целиком. К счастью, нигде в матричном дифференцировании, которое мы использовали выше, не требуется квадратность матриц, ведь матрица $C$ имеет размерность $n \times (n+m)$.

Выведите градиентный шаг для обновления $C$ в следующей клетке.

YOUR MARKDOWN BELOW

e = ...

...

$\hat{C} -= ...$

YOUR MARKDOWN ABOVE

### Задача 5

Реализуйте выведенный метод для осциллятора с управлением. На этот раз система будет колебаться не сама по себе, мы будем прикладывать к ней случайные воздействия, и так сможем оценить как матрицу самой системы $A$, так и матрицу управления $B$.

In [ ]:
def run_oscillator(x0, iter_num, visualize, m, real_stiffness, dt, compensation = False,
                 update = False):
    if (visualize == True):
        WIND_X = 700
        WIND_Y = 700
        canvas = np.ones((700, 700, 3), np.uint8) * 70
    
    #real A matrix
    A = np.array([[0, 1],
                  [- real_stiffness / m, -0.01]])
    
    #initial estimate
    C_hat = np.array([[0.0, 0, 0],
                      [0, 0, 0]])

    i = 0

    x_traj = []
    v_traj = []
    u_traj = []
    
    oscillator = Oscillator(A, x0 = x0, dt = dt, real_stiffness = real_stiffness)
    
    desired_x = 0
            
    while(True):
        state = copy.deepcopy(oscillator.get_state())
        
        control = np.random.randn(1)[0]
        
        x_dot = oscillator.sys_dyn(control)
                
        oscillator.integrate_sys_dyn(x_dot)
        
        # YOUR CODE BELOW
        
        # YOUR CODE ABOVE
        
        x_traj.append(state[0, 0])
        v_traj.append(state[1, 0])
        u_traj.append(control)
        
        i += 1

        if (i > iter_num):
            break

        if (visualize == True):
            canvas = cv2.addWeighted(canvas, 0.93, canvas, 0, 0)

            oscillator.draw(canvas, WIND_X // 2, WIND_Y // 2, 100,
                          state[0, 0], (123, 234, 45), inverted = False)

            cv2.imshow("oscillator", canvas)

            key = cv2.waitKey(10) & 0xFF

            if (key == ord('q')):
                break
        
    if (visualize == True):
        cv2.destroyAllWindows()
        cv2.waitKey(10)
    
    plot_1d([x_traj], "time", "x", "coordinate by time")
    plot_1d([v_traj], "time", "x_dot", "velocity by time")
    plot_1d([u_traj], "time", "F", "force by time")
    
    plt.figure(figsize=(10, 10))
    plt.plot(x_traj, v_traj)
    plt.show()
    clear_output(wait = True)
    
    return x_traj, v_traj, u_traj, C_hat, A

dt = 0.05

x_hist, v_hist, u_hist, C_hat, A = run_oscillator(x0 = -2, iter_num = 1200, visualize = False,
                        m = 1, real_stiffness = 0.9, dt = dt, compensation = True, update = True)

Найдите матрицы $A$ и $B$ непрерывной системы.

In [ ]:
# YOUR CODE BELOW



print(C_ha)

# YOUR CODE ABOVE

Попробуйте усилить шум, умножив его на число. Обратите внимание, что хоть траектория и становится все более и более хаотичной, осцилляции все же сохраняются. Как влияет увеличение амплитуды шума на сходимость параметров к реальному значению?

Попробуйте добавить в систему вязкое трение. Как это влияет на траекторию системы?

### Задача 6: адаптивное управление маятником

1) скопируйте в клетку ниже свое управление на основе энергии
2) добавьте в динамику системы вязкое трение
3) добавьте в цикл управления такую задержку, чтобы стабилизация на достигалась (обратите внимание, мы пытаемся сломать систему, усложняя задачу управления)

In [ ]:
import numpy as np
import math
import cv2
import matplotlib.pyplot as plt
from ipywidgets import interact
from IPython.display import display, clear_output

def plot_1d(data):
    plt.figure(figsize=(15, 4))
    plt.plot(data)
    plt.show()

class Pendulum:
    def __init__(self, m = 1.0, l = 1.0, g = 10.0, x0 = 3, v0 = 0.0, dt = 0.01):
        self.m = m
        self.l = l
        self.g = g
        
        self.x = np.array([[x0], [v0]])
        self.dt = dt
        
    def get_state(self):
        return self.x
    
    def sys_dyn(self, u_scalar):
        x_dot = np.zeros_like(self.x)
                
        x_dot[0] = self.x[1]
        x_dot[1] = self.g / self.l * np.sin(self.x[0]) + u_scalar / (self.m * self.l**2)
        
        return x_dot
    
    def integrate_sys_dyn(self, sys_dyn):        
        self.x[1] += sys_dyn[1] * self.dt
        self.x[0] += self.x[1] * self.dt

def draw_pendulum(canvas, x, y, l, alpha, color = (123, 234, 234),
                  inverted = False):
    direction = -1
    
    if (inverted == True):
        direction = 1
    
    endpoint = (int(x - l * np.cos(alpha + direction * np.pi / 2)),
                int(y + l * np.sin(alpha + direction * np.pi / 2)))
    
    cv2.line(canvas, (x, y), endpoint, (123, 234, 234), 5)
    
    cv2.circle(canvas, endpoint, 25, color, -1)

def swing_up_feedback(x, max_tau = 3):
    u = 0
    
    if (np.cos(x[0, 0]) > 0.7):
        u = - 30 * x[0, 0] - 14 * x[1, 0]
    
    else:
        u = 0.1 * x[1, 0]
    
    u = np.clip(u, -max_tau, max_tau)
    
    return u

def run_pendulum(x0, iter_num, visualize):
    if (visualize == True):
        WIND_X = 700
        WIND_Y = 700
        canvas = np.ones((700, 700, 3), np.uint8) * 70
    
    pendulum = Pendulum(x0 = x0)

    i = 0

    x_traj = []
    v_traj = []
    u_traj = []
    
    while(True):
        state = pendulum.get_state()

        control = swing_up_feedback(state, max_tau = 3)
        
        x_dot = pendulum.sys_dyn(control)
                
        pendulum.integrate_sys_dyn(x_dot)
        
        x_traj.append(state[0, 0])
        v_traj.append(state[1, 0])
        u_traj.append(control)
        
        i += 1

        if (i > iter_num):
            break

        if (visualize == True):
            canvas = cv2.addWeighted(canvas, 0.93, canvas, 0, 0)

            draw_pendulum(canvas, WIND_X // 2, WIND_Y // 2, 300,
                          state[0, 0], (123, 234, 45), inverted = False)

            cv2.imshow("pendulum", canvas)

            key = cv2.waitKey(10) & 0xFF

            if (key == ord('q')):
                break
    
    if (visualize == True):
        cv2.destroyAllWindows()
        cv2.waitKey(10)
    
    return x_traj, v_traj, u_traj

x_hist, v_hist, u_hist = run_pendulum(x0 = 3.0, iter_num = 10000, visualize = False)

plot_1d(x_hist)
plot_1d(v_hist)
plot_1d(u_hist)

plt.figure(figsize=(15, 15))
plt.plot(x_hist, v_hist)
plt.show()
clear_output(wait = True)

1: Компенсация

1) введите искусственное ограничение на момент в 0.8 max_torque
2) добавьте компенсацию трения
3) нарисуйте график момента от времени без компенсации и с компенсацией

2: Оценка

1) добавьте график оценки коэффициента трения и реального коэффициента. Оценка пока что может быть любой
2) добавьте обновление оценки коэффициента трения. Посмотрите на график. Достигается ли сходимость?
3) модифицируйте код так, чтобы оценка не затиралась после выполнения симуляции. Достигается ли сходимость, если запускать симуляцию много раз? Как улучшить сходимость?